# 03 — Kualitas Sinyal: Kapan Tampil 5 vs 10 Coin, Win Rate per Coin, Filter Kepercayaan

**Permintaan user**: rekomendasi spot **minimal 5, maksimal 10 coin** dengan win rate dan sinyal yang bagus.
Saat pasar risk-off tetap tampilkan 5-10 coin **dengan peringatan**. Riset dulu sebelum endpoint.

**Dasar** (notebook 01-02, tidak diubah): skor = rata-rata rank-percentile **RSI(14) + RSI(7)** antar coin;
masuk di open besok; exit = skor kemarin < **persentil 30%** ATAU **14 hari** ATAU **stop keras −25%**;
universe 40 coin (Large+Mid minus stablecoin/emas/WBETH), BTC tidak direkomendasikan.

### Pertanyaan (aturan & ambang ditetapkan SEBELUM melihat hasil uji)
| # | Pertanyaan | Kandidat (pilih di dev 2021–2023, laporkan di uji 2024–2026) |
|---|---|---|
| 0 | Reproduksi notebook 02 | top-5 risk-on uji harus = 45,5% / +1,79% |
| 1 | Peringkat 6–10 masih layak? | ekspektasi & win rate per kelompok peringkat 1–5, 6–10, 11–15 |
| 2 | Aturan "5 sampai 10" | peringkat 6–10 tampil hanya kalau skor ≥ ambang ∈ {0,80; 0,85; 0,90} |
| 3 | Win rate per coin bisa dipercaya? | apakah win rate coin di dev memprediksi win rate di uji; filter win rate berjalan 365 hari (≥5 sinyal selesai) < 35% dibuang |
| 4 | Filter kepercayaan | umur listing ≥ 180 hari; likuiditas rata-rata 20 hari ≥ $5 jt/hari; buang yang naik > 50% dalam 7 hari |
| 5 | Statistik saat risk-off | top-5 saat filter H8b mati (untuk peringatan) |
| 6 | Konsistensi per tahun | aturan final per tahun 2021–2026 |

Evaluasi = **per sinyal independen** (pelajaran notebook 02: simulasi slot portofolio bias seleksi).
Biaya 0,15%/sisi.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from data_source.fear_greed import load_fear_greed_index

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
RAW = PROJECT_ROOT / "notebooks" / "data" / "raw"
STABLE = {"BFUSDUSDT", "RLUSDUSDT", "USD1USDT", "USDCUSDT", "USDEUSDT", "USDSUSDT", "UUSDT"}
GOLD, DUP = {"PAXGUSDT", "XAUTUSDT"}, {"WBETHUSDT"}
ALL_SYMS = sorted(p.stem.replace("_1d_full", "") for p in RAW.glob("*_1d_full.csv"))
UNIV = [s for s in ALL_SYMS if s not in STABLE | GOLD | DUP]
BTC, COST = "BTCUSDT", 0.0015
COINS_NB = [c for c in UNIV if c != BTC]

def load(sym):
    df = pd.read_csv(RAW / f"{sym}_1d_full.csv", parse_dates=[0], index_col=0)
    df.columns = [c.lower() for c in df.columns]
    return df

raw = {s: load(s) for s in UNIV}
dates = pd.date_range("2021-01-01", "2026-06-30", freq="D")
close_df = pd.DataFrame({s: raw[s]["close"].reindex(dates) for s in UNIV})
open_df = pd.DataFrame({s: raw[s]["open"].reindex(dates) for s in UNIV})
low_df = pd.DataFrame({s: raw[s]["low"].reindex(dates) for s in UNIV})
vol_df = pd.DataFrame({s: raw[s]["volume"].reindex(dates) for s in UNIV})
DEV, TEST = ("2021-01-01", "2023-12-31"), ("2024-01-01", "2026-06-30")

fgi = load_fear_greed_index()["fng_value"].reindex(dates).ffill()
market_ok = ((close_df[BTC] > close_df[BTC].rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50))

def rsi(s, n=14):
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    return 100 - 100 / (1 + up / dn)

score_df = (close_df.apply(rsi, n=14).rank(axis=1, pct=True) + close_df.apply(rsi, n=7).rank(axis=1, pct=True)) / 2

# fitur "kepercayaan" (semua dihitung s/d hari sinyal, tanpa melihat masa depan)
first_valid = close_df.apply(lambda s: s.first_valid_index())
age_days = pd.DataFrame({s: (dates - first_valid[s]).days for s in UNIV}, index=dates)
liq_usd = (close_df * vol_df).rolling(20, min_periods=20).mean()          # rata-rata nilai transaksi/hari (USDT)
ret7 = close_df / close_df.shift(7) - 1
print(f"Universe {len(UNIV)} coin (kandidat rekomendasi {len(COINS_NB)}), {dates[0].date()} → {dates[-1].date()}")

Universe 40 coin (kandidat rekomendasi 39), 2021-01-01 → 2026-06-30


## Mesin evaluasi per sinyal

Untuk **setiap hari** dan **setiap peringkat 1–15** (risk-on maupun risk-off) dibuat satu sinyal independen:
masuk di open besok, keluar menurut aturan notebook 02. Semua fitur sinyal (peringkat, skor, filter pasar,
umur listing, likuiditas, kenaikan 7 hari) dicatat, lalu tiap pertanyaan memfilter tabel sinyal yang sama.

In [2]:
CLOSE, OPEN, LOW = (df[UNIV].to_numpy(float) for df in (close_df, open_df, low_df))
SCORE = score_df[UNIV].to_numpy(float)
NB_IDX = np.array([UNIV.index(c) for c in COINS_NB])
FLOOR, MAX_HOLD, HARD_STOP, MAX_RANK = 0.30, 14, 0.25, 15

rows = []
for i in range(len(dates) - 2):
    row = SCORE[i, NB_IDX]
    valid = ~np.isnan(row)
    if valid.sum() < 5:
        continue
    order = np.argsort(np.where(valid, row, -np.inf))[::-1][:min(MAX_RANK, valid.sum())]
    for rank, jj in enumerate(order, start=1):
        ci = NB_IDX[jj]
        entry = OPEN[i + 1, ci]
        if np.isnan(entry) or entry <= 0:
            continue
        stop = entry * (1 - HARD_STOP)
        exit_px, reason, exit_i = None, None, None
        end = min(i + 1 + MAX_HOLD, CLOSE.shape[0])
        for day in range(i + 1, end):
            lo, op = LOW[day, ci], OPEN[day, ci]
            if not np.isnan(lo) and lo <= stop:
                exit_px = op if (day != i + 1 and not np.isnan(op) and op <= stop) else stop
                reason, exit_i = "STOP_KERAS", day; break
            sc_y = SCORE[day - 1, ci]                       # skor KEMARIN (tanpa lookahead)
            if day > i + 1 and not np.isnan(sc_y) and sc_y < FLOOR:
                exit_px, reason, exit_i = (op if not np.isnan(op) else CLOSE[day, ci]), "SKOR_TURUN", day; break
        if exit_px is None:
            exit_i = min(end - 1, CLOSE.shape[0] - 1)
            exit_px = CLOSE[exit_i, ci]
            if np.isnan(exit_px):
                continue
            reason = "MAKS_HARI"
        coin = UNIV[ci]
        rows.append(dict(i=i, date=dates[i], coin=coin, rank=rank, score=row[jj], market_ok=bool(market_ok.iloc[i]),
                         age=age_days.iat[i, ci], liq=liq_usd.iat[i, ci], ret7=ret7.iat[i, ci],
                         exit_i=exit_i, reason=reason, ret=exit_px * (1 - COST) / (entry * (1 + COST)) - 1))
sig = pd.DataFrame(rows)
sig["year"] = sig["date"].dt.year
in_dev = (sig.date >= DEV[0]) & (sig.date < DEV[1])      # batas akhir eksklusif, sama dengan notebook 02
in_test = (sig.date >= TEST[0]) & (sig.date < TEST[1])
print(f"{len(sig):,} sinyal (hari × peringkat 1-{MAX_RANK})")

def summ(d):
    if len(d) < 20:
        return dict(n=len(d), win_rate=np.nan, rata2_untung=np.nan, rata2_rugi=np.nan, ekspektasi=np.nan)
    w = d.ret > 0
    return dict(n=len(d), win_rate=w.mean(), rata2_untung=d.ret[w].mean(), rata2_rugi=d.ret[~w].mean(), ekspektasi=d.ret.mean())

def fmt(df):
    out = df.copy()
    for c in ("win_rate", "rata2_untung", "rata2_rugi", "ekspektasi"):
        if c in out:
            out[c] = df[c].map(lambda v: f"{v:+.2%}" if c != "win_rate" and pd.notna(v) else (f"{v:.1%}" if pd.notna(v) else "-"))
    return out

29,865 sinyal (hari × peringkat 1-15)


## 0. Reproduksi notebook 02 (cek mesin)

In [3]:
r0 = summ(sig[in_test & sig.market_ok & (sig["rank"] <= 5)])
print(f"Top-5, risk-on, uji: n={r0['n']} win_rate={r0['win_rate']:.1%} ekspektasi={r0['ekspektasi']:+.2%}")
print("Notebook 02 mencatat : n=2855 win_rate=45.5% ekspektasi=+1.79%")

Top-5, risk-on, uji: n=2855 win_rate=45.5% ekspektasi=+1.79%
Notebook 02 mencatat : n=2855 win_rate=45.5% ekspektasi=+1.79%


## 1. Kualitas per kelompok peringkat (risk-on)

In [4]:
grp = pd.cut(sig["rank"], [0, 5, 10, 15], labels=["peringkat 1-5", "peringkat 6-10", "peringkat 11-15"])
rows = []
for label, mask in (("dev 2021-23", in_dev), ("uji 2024-26", in_test)):
    for g in grp.cat.categories:
        rows.append(dict(periode=label, kelompok=g, **summ(sig[mask & sig.market_ok & (grp == g)])))
fmt(pd.DataFrame(rows).set_index(["periode", "kelompok"]))

n win_rate rata2_untung rata2_rugi ekspektasi
periode     kelompok                                                         
dev 2021-23 peringkat 1-5    2780    46.4%      +33.13%    -12.09%     +8.88%
            peringkat 6-10   2780    47.5%      +26.43%    -10.56%     +7.00%
            peringkat 11-15  2780    42.9%      +18.83%     -7.87%     +3.59%
uji 2024-26 peringkat 1-5    2855    45.5%      +19.16%    -12.70%     +1.79%
            peringkat 6-10   2855    41.2%      +19.51%    -11.00%     +1.58%
            peringkat 11-15  2855    40.7%      +18.94%    -10.38%     +1.55%

## 2. Aturan "5 sampai 10": peringkat 6–10 hanya tampil kalau skor ≥ ambang

Ambang dipilih di **dev**: ambang terendah yang membuat peringkat 6–10 yang lolos punya ekspektasi
≥ ekspektasi peringkat 1–5 (artinya coin tambahan tidak menurunkan kualitas daftar).

In [5]:
base_dev = summ(sig[in_dev & sig.market_ok & (sig["rank"] <= 5)])
rows = []
for thr in (0.80, 0.85, 0.90):
    for label, mask in (("dev", in_dev), ("uji", in_test)):
        extra = sig[mask & sig.market_ok & sig["rank"].between(6, 10) & (sig.score >= thr)]
        days = sig[mask & sig.market_ok].date.nunique()
        rows.append(dict(ambang=thr, periode=label, **summ(extra),
                         rata2_coin_tambahan_per_hari=len(extra) / max(days, 1)))
t2 = pd.DataFrame(rows)
chosen_thr = next((thr for thr in (0.80, 0.85, 0.90)
                   if t2[(t2.ambang == thr) & (t2.periode == "dev")].ekspektasi.iloc[0] >= base_dev["ekspektasi"]), None)
print(f"Patokan dev peringkat 1-5: ekspektasi {base_dev['ekspektasi']:+.2%}, win rate {base_dev['win_rate']:.1%}")
print(f"Ambang terpilih dari dev: {chosen_thr}")
out = fmt(t2.set_index(["ambang", "periode"]))
out["rata2_coin_tambahan_per_hari"] = t2.set_index(["ambang", "periode"])["rata2_coin_tambahan_per_hari"].map(lambda v: f"{v:.1f}")
out

Patokan dev peringkat 1-5: ekspektasi +8.88%, win rate 46.4%
Ambang terpilih dari dev: None


n win_rate rata2_untung rata2_rugi ekspektasi rata2_coin_tambahan_per_hari
ambang periode                                                                              
0.80   dev      150    49.3%      +21.83%    -11.70%     +4.84%                          0.3
       uji      855    40.0%      +19.24%    -10.72%     +1.27%                          1.5
0.85   dev        5        -            -          -          -                          0.0
       uji      182    39.6%      +19.01%    -10.42%     +1.22%                          0.3
0.90   dev        0        -            -          -          -                          0.0
       uji        0        -            -          -          -                          0.0

## 3. Apakah win rate per coin bisa dipercaya?

(a) **Persistensi**: kalau sebuah coin win rate-nya tinggi di dev, apakah di uji juga tinggi?
(b) **Filter berjalan**: setiap sinyal, hitung win rate coin itu dari sinyal top-10 yang **sudah selesai**
dalam 365 hari terakhir (minimal 5 sinyal). Buang kalau < 35%. Bandingkan dengan tanpa filter.

In [6]:
base = sig[sig.market_ok & (sig["rank"] <= 10)]
wr_dev = base[in_dev].groupby("coin").ret.agg(lambda r: (r > 0).mean()).rename("wr_dev")
n_dev = base[in_dev].groupby("coin").size().rename("n_dev")
wr_test = base[in_test].groupby("coin").ret.agg(lambda r: (r > 0).mean()).rename("wr_uji")
n_test = base[in_test].groupby("coin").size().rename("n_uji")
pc = pd.concat([wr_dev, n_dev, wr_test, n_test], axis=1).dropna()
pc = pc[(pc.n_dev >= 20) & (pc.n_uji >= 20)]
rho, p = spearmanr(pc.wr_dev, pc.wr_uji)
top_half = pc.wr_dev >= pc.wr_dev.median()
print(f"(a) {len(pc)} coin dengan >=20 sinyal di dev & uji")
print(f"    korelasi peringkat win rate dev vs uji: {rho:+.2f} (p={p:.2f})")
print(f"    win rate uji coin separuh-atas dev: {pc.wr_uji[top_half].mean():.1%} | separuh-bawah dev: {pc.wr_uji[~top_half].mean():.1%}")

# (b) win rate berjalan tanpa lookahead: hanya sinyal yang exit_i < i
all10 = sig[sig["rank"] <= 10].sort_values("i")
trail = {}
for coin, g in all10.groupby("coin"):
    ii, ex, win = g.i.to_numpy(), g.exit_i.to_numpy(), (g.ret.to_numpy() > 0)
    out = np.full(len(g), np.nan)
    for k in range(len(g)):
        m = (ex < ii[k]) & (ii >= ii[k] - 365)
        if m.sum() >= 5:
            out[k] = win[m].mean()
    trail.update(dict(zip(g.index, out)))
sig["trail_wr"] = pd.Series(trail)
rows = []
for label, mask in (("dev", in_dev), ("uji", in_test)):
    b = sig[mask & sig.market_ok & (sig["rank"] <= 5)]
    rows.append(dict(periode=label, filter="tanpa filter", **summ(b)))
    rows.append(dict(periode=label, filter="buang win rate berjalan < 35%", **summ(b[~(b.trail_wr < 0.35)])))
fmt(pd.DataFrame(rows).set_index(["periode", "filter"]))

(a) 26 coin dengan >=20 sinyal di dev & uji
    korelasi peringkat win rate dev vs uji: +0.16 (p=0.42)
    win rate uji coin separuh-atas dev: 43.8% | separuh-bawah dev: 40.7%


n win_rate rata2_untung rata2_rugi ekspektasi
periode filter                                                                         
dev     tanpa filter                   2780    46.4%      +33.13%    -12.09%     +8.88%
        buang win rate berjalan < 35%  2275    45.3%      +32.70%    -12.17%     +8.16%
uji     tanpa filter                   2855    45.5%      +19.16%    -12.70%     +1.79%
        buang win rate berjalan < 35%  2391    45.8%      +17.60%    -12.54%     +1.28%

## 4. Filter kepercayaan (umur listing, likuiditas, pump ekstrem)

Tiap filter diuji sendiri lalu digabung. Diterima kalau ekspektasi **dev** naik tanpa memangkas sinyal
terlalu banyak; hasil uji dilaporkan apa adanya.

In [7]:
FILTERS = {
    "umur listing >= 180 hari": lambda d: d.age >= 180,
    "likuiditas 20h >= $5 jt/hari": lambda d: d.liq >= 5e6,
    "tidak naik > 50% dalam 7 hari": lambda d: ~(d.ret7 > 0.50),
}
rows = []
for label, mask in (("dev", in_dev), ("uji", in_test)):
    b = sig[mask & sig.market_ok & (sig["rank"] <= 5)]
    rows.append(dict(periode=label, filter="(tanpa filter)", **summ(b), sisa_sinyal=1.0))
    for name, f in FILTERS.items():
        k = b[f(b)]
        rows.append(dict(periode=label, filter=name, **summ(k), sisa_sinyal=len(k) / len(b)))
    allf = b[np.logical_and.reduce([f(b) for f in FILTERS.values()])]
    rows.append(dict(periode=label, filter="SEMUA filter", **summ(allf), sisa_sinyal=len(allf) / len(b)))
t4 = pd.DataFrame(rows).set_index(["periode", "filter"])
out = fmt(t4); out["sisa_sinyal"] = t4["sisa_sinyal"].map(lambda v: f"{v:.0%}")
out

n win_rate rata2_untung rata2_rugi ekspektasi sisa_sinyal
periode filter                                                                                     
dev     (tanpa filter)                 2780    46.4%      +33.13%    -12.09%     +8.88%        100%
        umur listing >= 180 hari       2063    43.6%      +16.18%    -10.59%     +1.07%         74%
        likuiditas 20h >= $5 jt/hari   2624    46.2%      +32.69%    -12.21%     +8.53%         94%
        tidak naik > 50% dalam 7 hari  2498    46.6%      +31.39%    -11.40%     +8.52%         90%
        SEMUA filter                   1847    44.0%      +15.56%    -10.46%     +0.98%         66%
uji     (tanpa filter)                 2855    45.5%      +19.16%    -12.70%     +1.79%        100%
        umur listing >= 180 hari       2736    46.1%      +19.24%    -12.38%     +2.19%         96%
        likuiditas 20h >= $5 jt/hari   2683    45.6%      +19.77%    -12.77%     +2.06%         94%
        tidak naik > 50% dalam 7 hari  2686    45.0%      +17.29%    -12.43%     +0.94%         94%
        SEMUA filter                   2414    45.8%      +17.86%    -12.13%     +1.60%         85%

## 5. Statistik saat risk-off (dasar peringatan)

In [8]:
rows = []
for label, mask in (("dev", in_dev), ("uji", in_test)):
    for state, m2 in (("risk-on", sig.market_ok), ("risk-off", ~sig.market_ok)):
        rows.append(dict(periode=label, pasar=state, **summ(sig[mask & m2 & (sig["rank"] <= 5)])))
fmt(pd.DataFrame(rows).set_index(["periode", "pasar"]))

n win_rate rata2_untung rata2_rugi ekspektasi
periode pasar                                                     
dev     risk-on   2780    46.4%      +33.13%    -12.09%     +8.88%
        risk-off  2620    40.8%      +16.19%    -13.26%     -1.23%
uji     risk-on   2855    45.5%      +19.16%    -12.70%     +1.79%
        risk-off  1695    42.8%      +11.83%    -11.32%     -1.40%

## 6. Per tahun (top-5 risk-on, aturan notebook 02)

In [9]:
rows = [dict(tahun=y, **summ(sig[(sig.year == y) & sig.market_ok & (sig["rank"] <= 5)])) for y in range(2021, 2027)]
rows_off = [dict(tahun=y, **summ(sig[(sig.year == y) & ~sig.market_ok & (sig["rank"] <= 5)])) for y in range(2021, 2027)]
a = fmt(pd.DataFrame(rows).set_index("tahun")); a.columns = pd.MultiIndex.from_product([["risk-on"], a.columns])
b = fmt(pd.DataFrame(rows_off).set_index("tahun")); b.columns = pd.MultiIndex.from_product([["risk-off"], b.columns])
pd.concat([a, b], axis=1)

risk-on                                             risk-off                                            
            n win_rate rata2_untung rata2_rugi ekspektasi        n win_rate rata2_untung rata2_rugi ekspektasi
tahun                                                                                                         
2021     1220    52.2%      +53.11%    -15.63%    +20.26%      535    32.9%      +22.77%    -16.72%     -3.73%
2022      130    14.6%       +6.59%    -14.14%    -11.11%     1695    38.1%      +13.73%    -13.00%     -2.83%
2023     1435    44.3%      +13.83%     -9.20%     +0.99%      390    63.8%      +17.90%     -6.40%     +9.11%
2024     1565    47.6%      +24.46%    -13.43%     +4.61%      265    58.5%      +15.95%     -6.16%     +6.77%
2025     1065    44.0%      +11.39%    -11.85%     -1.62%      760    41.4%      +10.59%    -13.13%     -3.30%
2026      225    37.3%      +15.57%    -12.07%     -1.75%      670    38.2%      +10.86%    -10.74%     -2.48%

## Kesimpulan

**0. Mesin valid** — top-5 risk-on periode uji identik dengan notebook 02 (n=2855, win rate 45,5%, ekspektasi +1,79%).

**1. Peringkat 6–10 layak ditampilkan, tapi sebagai kelas kedua.**
| Uji 2024–26 (risk-on) | Win rate | Ekspektasi |
|---|---|---|
| Peringkat 1–5 | **45,5%** | +1,79% |
| Peringkat 6–10 | 41,2% | +1,58% |
| Peringkat 11–15 | 40,7% | +1,55% |

Ekspektasi 6–10 hampir sama dengan 1–5 (positif), tapi win rate-nya ~4 poin lebih rendah. Di periode uji
pemisahan antar peringkat **lemah** — keunggulan utamanya "coin ber-RSI relatif tinggi saat pasar risk-on",
bukan urutan persis peringkat 1 vs 8.

**2. Aturan "6–10 hanya tampil kalau skor ≥ ambang" DITOLAK.** Tidak ada ambang (0,80/0,85/0,90) yang di dev
membuat peringkat 6–10 setara peringkat 1–5; skor peringkat 6–10 nyaris tidak pernah ≥ 0,85 (skor adalah
persentil antar 40 coin, jadi peringkat 6 otomatis ≈ 0,85). → **Tampilkan selalu 10 coin**: 5 "utama" + 5
"pelengkap", masing-masing dengan statistik kelompoknya sendiri.

**3. Win rate per coin TIDAK bisa dipercaya.** Korelasi peringkat win rate coin dev vs uji hanya +0,16 (p=0,42,
tidak signifikan, 26 coin). Filter win rate berjalan < 35% malah menurunkan ekspektasi di dev (+8,88 → +8,16%)
maupun uji (+1,79 → +1,28%). → Jangan tampilkan "win rate coin X = 70%" seolah prediksi; yang valid adalah
**win rate kelompok peringkat** (tabel di atas).

**4. Filter kepercayaan — tidak ada yang lolos aturan dev.** Umur listing ≥ 180 hari membanting ekspektasi dev
(+8,88 → +1,07%, karena reli coin baru 2021); likuiditas ≥ $5 jt dan anti-pump juga sedikit menurunkan dev.
Likuiditas sedikit membantu di uji (+2,06%), tapi sesuai aturan yang ditetapkan sebelumnya **tidak diadopsi**
sebagai filter skor. Likuiditas minimum tetap masuk akal sebagai syarat **praktis** (spread/slippage modal
kecil), bukan karena terbukti menaikkan win rate.

**5. Risk-off → peringatan wajib.** Top-5 saat filter H8b mati: ekspektasi **negatif** di dev (−1,23%) dan uji
(−1,40%), win rate 41–43%. Daftar tetap 5–10 coin (keputusan user) tapi harus berlabel "pasar risk-off —
secara historis rata-rata sinyal rugi".

**6. ⚠️ Peringatan paling penting: edge melemah di data terbaru.** Risk-on per tahun: 2024 +4,61%, tapi
**2025 −1,62%** dan **2026 (s/d Juni) −1,75%** (win rate 37%). Angka gabungan uji +1,79% ditopang 2024.
Sinyal ini belum terbukti menguntungkan di 18 bulan terakhir.

**Jumlah percobaan**: 3 ambang + 1 filter win rate + 3 filter kepercayaan + 1 gabungan = 8 varian; **tidak ada
satu pun yang diadopsi**, jadi tidak ada inflasi akibat pemilihan (aturan dasar tetap = notebook 02).

### Rekomendasi desain (belum diimplementasi)
1. Selalu 10 coin: peringkat 1–5 "utama" (win rate historis ~45%), 6–10 "pelengkap" (~41%).
2. Tampilkan statistik **kelompok**, bukan win rate per coin.
3. Label status pasar (risk-on/off) + peringatan saat risk-off.
4. Sebelum dipakai uang nyata: **paper trading** dulu, karena 2025–2026 ekspektasinya negatif.